# 10b — Selected-model beeswarm and waterfall gallery

This companion notebook **loads existing Monte Carlo Shapley attributions** for the six headline configurations. It does not fit models or recompute attributions. It revalidates saved predictors against OOF probabilities to retrieve the corresponding held-out feature values.

All output classes are exported to PNG at 300 dpi. By default, only stress-class beeswarms and S2 waterfall examples, plus six global summaries, are embedded: **18 previews**, distributed across all six configurations. Other class plots remain available as PNG files. Change `INLINE_MODE` to `all` for every class, or `none` to keep the notebook light. Set `WATERFALL_SUBJECTS=None` to export individual examples for every subject.

This notebook uses saved attribution files under `artifacts/revision_v2/interpretability/selected_headline_probability_pairs_32/`. A separate `visual_gallery/` directory stores its figures. No SHAP or Numba import is required.


In [ ]:
from pathlib import Path
import os,sys,json,ast,re,itertools,warnings,hashlib
from io import BytesIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from scipy.stats import spearmanr
from sklearn.base import BaseEstimator,TransformerMixin
import joblib
from IPython.display import display,Image,Markdown

ARTIFACTS_REL='artifacts/revision_v2'
PROJECT_ROOT=None
here=Path.cwd().resolve()
ROOT=Path(PROJECT_ROOT).resolve() if PROJECT_ROOT else next(
    (p for p in [here,*here.parents] if (p/'configs/config.yaml').is_file() and (p/'src').is_dir()),None)
if ROOT is None:raise RuntimeError('Open this notebook in the project notebooks directory.')
os.chdir(ROOT)
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from src.fusion import comparison_tables
from src.exploratory_fusion import mask_modality
A=ROOT/ARTIFACTS_REL
PAIRS=32;BACKGROUND_PER_SUBJECT=10;MAX_BACKGROUND=150;BATCH_SIZE=4096;SEED=42
MAX_INLINE_FIGURES=18
INLINE_FIGURES_SHOWN=0
SHOW_FIGURES=True;TOP_N=15;BEESWARM_WINDOWS_PER_SUBJECT=20
WATERFALL_SUBJECTS=[2];WATERFALL_WINDOW_ID=None
LOW_CORRELATION=.30;HIGH_CORRELATION=.70;EFFECT_TOP_N=3;ALE_BINS=6;PDP_POINTS=15
RUN_EFFECT_PLOTS=True
OUT=A/'interpretability'/f'selected_headline_probability_pairs_{PAIRS}'
OUT.mkdir(parents=True,exist_ok=True)
CLASS_NAMES={0:'Non-stress',1:'Baseline',2:'Stress',3:'Amusement',4:'Meditation'}
def class_name(task,k):
    return {0:'Non-stress',1:'Stress'}[int(k)] if task=='stress_vs_nonstress' else CLASS_NAMES.get(int(k),str(k))
def safe(s):return re.sub(r'[^a-zA-Z0-9_.-]+','_',str(s))
def pretty(s):return s.replace('__',' · ').replace('_',' ')
SOURCE=OUT
GALLERY=SOURCE/'visual_gallery'
GALLERY.mkdir(parents=True,exist_ok=True)
INLINE_MODE='stress_only'  # 'stress_only', 'all', or 'none'
WATERFALL_SUBJECTS=[2]  # Set to None to export a waterfall for every participant.
WATERFALL_WINDOW_ID=None
TOP_N=12
figure_inventory=[]
def show(fig,path):
    relative=path.relative_to(SOURCE)
    destination=GALLERY/relative
    destination.parent.mkdir(parents=True,exist_ok=True)
    fig.savefig(destination,dpi=300,bbox_inches='tight')
    kind=relative.parts[0]
    stress_class=1 if 'stress_vs_nonstress' in destination.name else 2
    is_stress=f'_class{stress_class}.' in destination.name
    inline=(INLINE_MODE=='all' or (INLINE_MODE=='stress_only' and (kind=='summary' or is_stress)))
    figure_inventory.append(dict(kind=kind,figure=relative.as_posix(),displayed=inline))
    if inline:
        with BytesIO() as stream:
            fig.savefig(stream,format='png',dpi=85,bbox_inches='tight')
            display(Image(data=stream.getvalue()))
    plt.close(fig)

# Restore only the trusted selector definition; do not execute notebook09 training.
for name in ['09_feature_redundancy_and_tree_importance_v2.ipynb','09_feature_redundancy_and_tree_importance.ipynb']:
    nbpath=ROOT/'notebooks'/name
    if not nbpath.is_file():continue
    book=json.loads(nbpath.read_text(encoding='utf-8'))
    for cell in book['cells']:
        if cell['cell_type']=='code' and 'class CorrelationSelector' in ''.join(cell['source']):
            parsed=ast.parse(''.join(cell['source']))
            node=next(n for n in parsed.body if isinstance(n,ast.ClassDef) and n.name=='CorrelationSelector')
            exec(compile(ast.Module(body=[node],type_ignores=[]),'<saved selector>','exec'),globals())
            break
    if 'CorrelationSelector' in globals():break
print('Visual gallery configured: saved attributions only; no retraining or Shapley recomputation.')


## Validate the six selected model configurations

In [ ]:
registry=[];missing=[]
result_roots=[A/'results',A/'exploratory/results',A/'exploratory/four_class/results']
for result_root in result_roots:
    for path in sorted(result_root.glob('*.json')):
        result=json.loads(path.read_text(encoding='utf-8'))
        if not isinstance(result,dict) or not all(k in result for k in ['task','model','site','classes','folds']):continue
        experiment=path.stem;parent=result_root.parent
        oof=path.with_name(path.stem+'_predictions.csv')
        method=result.get('method');scenario=result.get('scenario','both_present')
        for fold in result['folds']:
            sid=int(fold['subject_id'])
            folder=parent/'models'/(method if method in ['late_fusion','modality_dropout'] else f"{result['task']}_{result['site']}_{result['model']}")
            model_file=folder/f'S{sid}.joblib';meta=folder/f'S{sid}_metadata.json'
            absent=[]
            if not model_file.is_file():absent.append('model')
            if not oof.is_file():absent.append('OOF')
            if not method and not meta.is_file():absent.append('metadata')
            row=dict(experiment=experiment,task=result['task'],site=result['site'],model=result['model'],variant='full',
                subject_id=sid,classes=result['classes'],method=method,scenario=scenario,model_file=model_file,
                metadata_file=meta,prediction_file=oof,feature_root=parent/'features' if result['task']=='four_class' else A/'features')
            if absent:missing.append({**{k:row[k] for k in ['experiment','task','site','model','variant','subject_id']},'reason':'missing '+', '.join(absent)})
            else:registry.append(row)

filter_root=A/'exploratory/feature_redundancy_saved_models'
if not filter_root.is_dir():filter_root=A/'exploratory/feature_redundancy_importance'
metrics_path=filter_root/'fold_metrics.csv'
if metrics_path.is_file():
    metric=pd.read_csv(metrics_path)
    for r in metric[['site','model','variant','subject_id']].drop_duplicates().itertuples(index=False):
        sid=int(r.subject_id);stem=f'{r.site}_{r.model}_{r.variant}_S{sid}'
        model_file=filter_root/f'{stem}.joblib';meta=filter_root/f'{stem}_metadata.json';oof=filter_root/'predictions.csv'
        absent=[name for name,p in [('model',model_file),('metadata',meta),('OOF',oof)] if not p.is_file()]
        experiment=f'sensitivity_{r.site}_{r.model}_{r.variant}'
        if absent:missing.append(dict(experiment=experiment,task='three_class',site=r.site,model=r.model,variant=r.variant,subject_id=sid,reason='missing '+', '.join(absent)))
        else:registry.append(dict(experiment=experiment,task='three_class',site=r.site,model=r.model,variant=r.variant,
            subject_id=sid,classes=[1,2,3],method=None,scenario='both_present',model_file=model_file,
            metadata_file=meta,prediction_file=oof,feature_root=A/'features'))
if (A/'results/uncertainty_predictions.csv').is_file():
    missing.append(dict(experiment='uncertainty_calibrated',reason='fitted estimators/calibrators not persisted by uncertainty workflow'))
registry=pd.DataFrame(registry)
pd.DataFrame(missing).to_csv(OUT/'unavailable_experiments.csv',index=False)
if registry.empty:raise RuntimeError('No complete saved experiments found. Check ARTIFACTS_REL.')
SELECTED_EXPERIMENTS=['confirmatory_wrist_random_forest', 'confirmatory_early_fusion_logistic_regression', 'late_fusion_both_present', 'sensitivity_early_fusion_random_forest_reduced_0.90', 'secondary_stress_vs_nonstress_wrist', 'secondary_baseline_vs_stress_wrist']
registry=registry[registry.experiment.isin(SELECTED_EXPERIMENTS)].copy()
missing=[r for r in missing if r.get('experiment') in SELECTED_EXPERIMENTS]
expected_subjects={2,3,4,5,6,7,8,9,10,11,13,14,15,16,17}
coverage=[]
for experiment in SELECTED_EXPERIMENTS:
    part=registry[registry.experiment.eq(experiment)]
    observed=set(part.subject_id.astype(int))
    coverage.append(dict(experiment=experiment,available_folds=len(part),complete=(observed==expected_subjects and len(part)==15)))
coverage=pd.DataFrame(coverage)
display(coverage)
pd.DataFrame(missing).to_csv(OUT/'unavailable_experiments.csv',index=False)
if not coverage.complete.all():
    raise RuntimeError('Selected-model coverage incomplete. Restore the listed saved models, metadata and OOF predictions. No replacement training is performed.')
registry=registry.sort_values(['experiment','subject_id']).reset_index(drop=True)
registry.drop(columns=['model_file','metadata_file','prediction_file','feature_root']).to_csv(OUT/'model_registry.csv',index=False)


In [ ]:
frame_cache={}
def get_frames(root):
    key=str(root)
    if key not in frame_cache:
        tables={s:pd.read_parquet(root/f'{s}_features.parquet') for s in ['chest','wrist']}
        for frame in tables.values():
            if frame.valid.dtype!=bool:
                text=frame.valid.astype(str).str.lower()
                if not text.isin(['true','false']).all():raise ValueError('Invalid validity flags.')
                frame['valid']=text.eq('true')
        paired=comparison_tables(tables,True);available=comparison_tables(tables,False)
        frame_cache[key]=(paired,available)
    return frame_cache[key]

def prepare(r):
    paired,available=get_frames(r.feature_root)
    frame=(available[r.site] if r.task=='availability_three_class' else
           paired['early_fusion'] if r.method in ['late_fusion','modality_dropout'] else paired[r.site]).copy()
    if r.task=='stress_vs_nonstress':frame['label']=frame.label.eq(2).astype(int)
    elif r.task=='baseline_vs_stress':frame=frame[frame.label.isin([1,2])].copy()
    else:frame=frame[frame.label.isin(r.classes)].copy()
    artifact=joblib.load(r.model_file)
    if r.method in ['late_fusion','modality_dropout']:
        fit_subjects=list(map(int,artifact['fit_subjects']))
        bundle=artifact['bundle']
        if r.method=='late_fusion':
            cols=sorted(set().union(*(set(bundle[s]['features']) for s in ['chest','wrist'])))
            weights=artifact['metadata']['fixed_probability_weights']
            def predict(X):
                outputs=[]
                for s in ['chest','wrist']:
                    m=bundle[s]['model']
                    if list(map(int,m.classes_))!=list(r.classes):raise ValueError('Composite class order mismatch.')
                    outputs.append(weights[s]*m.predict_proba(X[bundle[s]['features']]))
                return sum(outputs)
        else:
            model=bundle['model'];stored=bundle['features']
            cols=[col for col in stored if '__available' not in col]
            if list(map(int,model.classes_))!=list(r.classes):raise ValueError('Composite class order mismatch.')
            def predict(X):return model.predict_proba(mask_modality(X,r.scenario)[stored])
    else:
        meta=json.loads(r.metadata_file.read_text(encoding='utf-8'))
        if int(meta['test_subject'])!=r.subject_id:raise ValueError('Metadata test participant mismatch.')
        fit_subjects=list(map(int,meta['fit_subjects']));cols=meta['features'];model=artifact
        if list(map(int,model.classes_))!=list(r.classes):raise ValueError('Class order mismatch.')
        def predict(X):return model.predict_proba(X[cols])
    if r.subject_id in fit_subjects or set(fit_subjects)!=set(frame.subject_id.unique())-{r.subject_id}:
        raise ValueError('Invalid LOSO train/test provenance.')
    train=frame[frame.subject_id.isin(fit_subjects)].copy()
    test=frame[frame.subject_id.eq(r.subject_id)].copy()
    oof=pd.read_csv(r.prediction_file)
    if r.experiment.startswith('sensitivity_'):
        oof=oof[(oof.site==r.site)&(oof.model==r.model)&(oof.variant==r.variant)]
    oof=oof[oof.subject_id.eq(r.subject_id)]
    if oof.window_id.duplicated().any() or set(oof.window_id)!=set(test.window_id):raise ValueError('OOF windows mismatch.')
    test=test.set_index('window_id').loc[oof.window_id].reset_index()
    if not np.array_equal(test.label.to_numpy(),oof.label.to_numpy()):raise ValueError('OOF labels mismatch.')
    expected=oof[[f'p_{k}' for k in r.classes]].to_numpy(float)
    if not np.allclose(predict(test[cols]),expected,rtol=1e-5,atol=1e-6):raise ValueError('Model does not reproduce saved OOF probabilities.')
    background=pd.concat([g.sample(min(len(g),BACKGROUND_PER_SUBJECT),random_state=SEED) for _,g in train.groupby('subject_id')])
    if len(background)>MAX_BACKGROUND:background=background.sample(MAX_BACKGROUND,random_state=SEED)
    return predict,train,test,cols,background[cols]

def shapley_sampling(predict,X,background,n_pairs=PAIRS,seed=SEED):
    if n_pairs<2:raise ValueError('At least two sampling pairs required.')
    rng=np.random.default_rng(seed);n,p=X.shape;actual=np.asarray(predict(X),float);k=actual.shape[1]
    mean=np.zeros((n,p,k));m2=np.zeros_like(mean);base_total=np.zeros(k)
    for iteration in range(n_pairs):
        reference=background.iloc[[rng.integers(len(background))]]
        base_total+=np.asarray(predict(reference),float)[0]
        permutation=rng.permutation(p);contribution=np.zeros_like(mean)
        for order in [permutation,permutation[::-1]]:
            state=pd.DataFrame(np.repeat(reference.to_numpy(copy=True),n,axis=0),columns=X.columns)
            states=[state.copy(deep=True)]
            for j in order:
                state.iloc[:,j]=X.iloc[:,j].to_numpy(copy=True);states.append(state.copy(deep=True))
            stack=pd.concat(states,ignore_index=True)
            output=np.concatenate([np.asarray(predict(stack.iloc[a:a+BATCH_SIZE]),float) for a in range(0,len(stack),BATCH_SIZE)])
            increments=np.diff(output.reshape(p+1,n,k),axis=0)
            for step,j in enumerate(order):contribution[:,j,:]+=.5*increments[step]
        delta=contribution-mean;mean+=delta/(iteration+1);m2+=delta*(contribution-mean)
    base=np.tile(base_total/n_pairs,(n,1));error=float(np.max(np.abs(base+mean.sum(1)-actual)))
    if not np.allclose(base+mean.sum(1),actual,atol=2e-5,rtol=2e-5):raise ValueError('Shapley additivity check failed.')
    se=np.sqrt(np.maximum(m2/(n_pairs-1),0)/n_pairs)
    return mean,base,se,error

def ale(predict,X,feature,edges):
    values=X[feature].to_numpy(float);bins=np.clip(np.searchsorted(edges,values,side='right')-1,0,len(edges)-2)
    eligible=np.isfinite(values)&(values>=edges[0])&(values<=edges[-1]);counts=[];increments=[]
    for i in range(len(edges)-1):
        rows=X.loc[eligible&(bins==i)].copy()
        if rows.empty:return None
        lo=rows.copy();hi=rows.copy();lo[feature]=edges[i];hi[feature]=edges[i+1]
        counts.append(len(rows));increments.append((predict(hi)-predict(lo)).mean(0))
    increments=np.asarray(increments);effects=np.cumsum(increments,axis=0)-increments/2
    effects-=np.average(effects,axis=0,weights=counts)
    return (edges[:-1]+edges[1:])/2,effects,counts

def family(feature):
    return {'ECG':'Cardiac ECG/HRV','BVP':'Pulse BVP/PRV','EDA':'Electrodermal','ACC':'Movement',
            'Temp':'Temperature','TEMP':'Temperature','Resp':'Respiration','EMG':'Muscle'}.get(feature.split('__')[1],'Other')


In [ ]:
required=['shap_feature_importance_by_fold.csv','shap_group_importance.csv','execution_status.csv']
if any(not (SOURCE/name).is_file() for name in required):
    raise RuntimeError('Run the selected-model attribution notebook first: saved attribution tables are required.')
executed=pd.read_csv(SOURCE/'execution_status.csv')
complete=executed[executed.status.eq('completed')]
expected=set(zip(registry.experiment,registry.subject_id.astype(int)))
actual=set(zip(complete.experiment,complete.subject_id.astype(int)))
if expected-actual:
    raise RuntimeError('Some selected folds have no completed attribution record. Complete the attribution notebook first.')
absent=[]
for r in registry.itertuples():
    file=SOURCE/'folds'/f'{r.experiment}_S{r.subject_id}'/f'shap_values_fold_{r.subject_id}.parquet'
    if not file.is_file():absent.append((r.experiment,r.subject_id))
if absent:
    display(pd.DataFrame(absent,columns=['experiment','subject_id']))
    raise RuntimeError('Saved per-window attribution files are missing.')
print('Saved attribution coverage verified: six configurations, 90 folds.')


## Global summaries, beeswarm and waterfall plots

Beeswarm points are sampled equally across test participants. Colors represent within-participant raw-feature percentiles. Waterfalls explain the first saved window of S2 by default; this deterministic example is not chosen for prediction quality and is not representative of the whole participant. All class probabilities have their own attribution problem.

Signed contributions increase or decrease the displayed class probability relative to the training reference. Contributions for features outside the displayed top set are added into an “other features” term so the waterfall retains its full sum.


In [ ]:
features=pd.read_csv(OUT/'shap_feature_importance_by_fold.csv')
groups=pd.read_csv(OUT/'shap_group_importance.csv')
execution=pd.read_csv(OUT/'execution_status.csv')
success=execution[execution.status.eq('completed')][['experiment','subject_id']].drop_duplicates()
stability=[];variant_comparison=[]
for experiment,g in features.groupby('experiment'):
    per_subject=g.groupby(['subject_id','feature']).mean_abs_shap.mean().unstack(fill_value=0)
    for a,b in itertools.combinations(per_subject.index,2):
        x,y=per_subject.loc[a],per_subject.loc[b]
        topx=set(x.nlargest(min(10,len(x))).index);topy=set(y.nlargest(min(10,len(y))).index)
        stability.append(dict(experiment=experiment,subject_a=a,subject_b=b,
            rank_spearman=float(spearmanr(x,y).statistic) if x.nunique()>1 and y.nunique()>1 else np.nan,
            top10_jaccard=len(topx&topy)/len(topx|topy)))
    importance=per_subject.mean().sort_values(ascending=False).head(TOP_N).sort_values()
    fig,ax=plt.subplots(figsize=(11,max(5,len(importance)*.35)))
    ax.barh([pretty(x) for x in importance.index],importance.values)
    ax.set(title=experiment,xlabel='Equal-participant, equal-class mean |Shapley| (probability)')
    fig.tight_layout();show(fig,OUT/'summary'/f'shap_summary_{safe(experiment)}.png')
    entries=registry[registry.experiment.eq(experiment)]
    saved=[]
    for r in entries.itertuples():
        if not ((success.experiment==experiment)&(success.subject_id==r.subject_id)).any():continue
        folder=OUT/'folds'/f'{experiment}_S{r.subject_id}'
        table=pd.read_parquet(folder/f'shap_values_fold_{r.subject_id}.parquet')
        # Revalidate provenance/probabilities; prepare() never fits.
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            predict,train,test,cols,bg=prepare(r)
        raw=test.set_index('window_id')[cols]
        colors=raw.rank(pct=True)
        saved.append((r,table,cols,colors))
    if not saved:continue
    for k in saved[0][0].classes:
        count=min(BEESWARM_WINDOWS_PER_SUBJECT,min(len(t[t.class_label.eq(k)]) for _,t,_,_ in saved))
        parts=[];colorparts=[]
        for r,t,cols,colors in saved:
            t=t[t.class_label.eq(k)]
            sampled=t.sample(count,random_state=SEED+r.subject_id)
            parts.append(sampled[cols].reset_index(drop=True))
            colorparts.append(colors.loc[sampled.window_id,cols].reset_index(drop=True))
            if WATERFALL_SUBJECTS is None or r.subject_id in WATERFALL_SUBJECTS:
                row=t.iloc[0] if WATERFALL_WINDOW_ID is None else t[t.window_id.eq(WATERFALL_WINDOW_ID)].iloc[0]
                v=row[cols].astype(float);top=list(v.abs().nlargest(TOP_N).index)
                names=[pretty(f) for f in top];increments=list(v[top]);rest=[f for f in cols if f not in top]
                if rest:names.append(f'Other {len(rest)} features');increments.append(float(v[rest].sum()))
                current=float(row.base_value);final=current+float(v.sum())
                fig,ax=plt.subplots(figsize=(12,max(5,len(names)*.35)))
                for i,(name,increment) in enumerate(zip(names,increments)):
                    end=current+increment
                    ax.barh(i,abs(increment),left=min(current,end),color='firebrick' if increment>=0 else 'steelblue')
                    ax.text((current+end)/2,i,f'{increment:+.3g}',va='center',ha='center',fontsize=8);current=end
                ax.axvline(float(row.base_value),color='gray',linestyle='--',label=f'Reference {float(row.base_value):.3f}')
                ax.axvline(final,color='black',label=f'Output {final:.3f}')
                ax.set(yticks=range(len(names)),yticklabels=names,title=f'{experiment} · S{r.subject_id} · {class_name(r.task,k)}',xlabel='Predicted probability')
                ax.invert_yaxis();ax.legend();fig.tight_layout()
                show(fig,OUT/'waterfalls'/f'{safe(experiment)}_S{r.subject_id}_class{k}.png')
        all_values=pd.concat(parts,ignore_index=True).fillna(0);all_colors=pd.concat(colorparts,ignore_index=True).reindex(columns=all_values.columns)
        top=all_values.abs().mean().nlargest(TOP_N).index[::-1]
        fig,ax=plt.subplots(figsize=(12,max(5,len(top)*.4)));rng=np.random.default_rng(SEED)
        for i,f in enumerate(top):
            values=all_values[f].to_numpy();color=all_colors[f].to_numpy();finite=np.isfinite(color)
            jitter=rng.uniform(-.28,.28,len(values))
            ax.scatter(values[finite],i+jitter[finite],c=color[finite],cmap='coolwarm',vmin=0,vmax=1,s=10,alpha=.6)
            if (~finite).any():ax.scatter(values[~finite],i+jitter[~finite],color='gray',s=10,alpha=.5)
        ax.axvline(0,color='black',linewidth=.8)
        ax.set(yticks=range(len(top)),yticklabels=[pretty(f) for f in top],title=f'{experiment} · {class_name(saved[0][0].task,k)}',xlabel='Approximate Shapley contribution (probability)')
        fig.colorbar(ScalarMappable(norm=Normalize(0,1),cmap='coolwarm'),ax=ax,label='Raw feature percentile within test participant')
        fig.tight_layout();show(fig,OUT/'beeswarm'/f'{safe(experiment)}_class{k}.png')

stability=pd.DataFrame(stability);stability.to_csv(GALLERY/'shap_rank_stability.csv',index=False)
# Compare variants only within the same task, site and algorithm, on common participants.
for (task,site,model),g in features.groupby(['task','site','model']):
    full=g[g.variant.eq('full') & ~g.experiment.str.startswith('sensitivity_')]
    for variant in [v for v in g.variant.unique() if v!='full']:
        reduced=g[g.variant.eq(variant)]
        for sid in sorted(set(full.subject_id)&set(reduced.subject_id)):
            x=full[full.subject_id.eq(sid)].groupby('feature').mean_abs_shap.mean()
            y=reduced[reduced.subject_id.eq(sid)].groupby('feature').mean_abs_shap.mean()
            union=x.index.union(y.index);x=x.reindex(union,fill_value=0);y=y.reindex(union,fill_value=0)
            variant_comparison.append(dict(task=task,site=site,model=model,variant=variant,subject_id=sid,
                rank_spearman=float(spearmanr(x,y).statistic) if x.nunique()>1 and y.nunique()>1 else np.nan))
pd.DataFrame(variant_comparison).to_csv(GALLERY/'full_reduced_comparison.csv',index=False)
display(stability.groupby('experiment')[['rank_spearman','top10_jaccard']].mean() if not stability.empty else stability)


In [ ]:
pd.DataFrame(figure_inventory).to_csv(GALLERY/'figure_inventory.csv',index=False)
display(pd.DataFrame(figure_inventory).groupby('kind').agg(saved=('figure','size'),displayed=('displayed','sum')).reset_index())
print('All figures saved at 300 dpi; only the requested previews are embedded.')


## Interpretation

A beeswarm shows the distribution and sign of model contributions, not a causal relationship. The color scale is relative within participant; a red point is not necessarily a clinically high value. Waterfalls are individual examples; their reference probability and contributions depend on the fitted fold and sampled training background.

These plots reuse approximate probability-scale Shapley values, not exact TreeSHAP. Monte Carlo uncertainty, correlated-feature attribution and implausible hybrid inputs remain relevant. They do not change model performance or support promotion of a post-hoc configuration to the primary result.
